In [ ]:
# Task 1: Feature engineering 
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Load the dataset
df = pd.read_csv("bike_rental_features.csv")

#Create interaction features between numerical columns
numerical_cols = [
    'Temperature(°C)', 'Humidity(%)', 'Wind speed (m/s)', 'Visibility (10m)',
    'Dew point temperature(°C)', 'Solar Radiation (MJ/m2)', 'Rainfall(mm)', 'Snowfall (cm)'
]

# Generate pairwise interaction features
for col1, col2 in combinations(numerical_cols, 2):
    df[f'{col1}*{col2}'] = df[col1] * df[col2]

# Drop rows with missing values (if any)
df.dropna(inplace=True)

# Scale numerical features
# Define the columns to scale: numerical + interaction features only
exclude_cols = [
    'Rented Bike Count', 'Seasons_Spring', 'Seasons_Summer', 'Seasons_Winter',
    'Holiday_No Holiday', 'Functioning Day_Yes'
]
cols_to_scale = [col for col in df.columns if col not in exclude_cols]

# Apply StandardScaler
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# ave the processed dataset 
df.to_csv("bike_rental_features.csv", index=False)

# Display first few rows of the final processed dataframe
df.head()


,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons_Spring,Seasons_Summer,...,Visibility (10m)*Dew point temperature(°C),Visibility (10m)*Solar Radiation (MJ/m2),Visibility (10m)*Rainfall(mm),Visibility (10m)*Snowfall (cm),Dew point temperature(°C)*Solar Radiation (MJ/m2),Dew point temperature(°C)*Rainfall(mm),Dew point temperature(°C)*Snowfall (cm),Solar Radiation (MJ/m2)*Rainfall(mm),Solar Radiation (MJ/m2)*Snowfall (cm),Rainfall(mm)*Snowfall (cm)
0,-1.513957,-1.042483,0.458476,0.925871,-1.659605,-0.655132,-0.1318,-0.171891,0.0,0.0,...,-1.556426,-0.883135,0.029629,-0.025941,1.119324,0.079541,0.559114,0.266319,0.311237,0.009676
1,-1.539074,-0.993370,-0.892561,0.925871,-1.659605,-0.655132,-0.1318,-0.171891,0.0,0.0,...,-1.556426,-0.883135,0.029629,-0.025941,1.119324,0.079541,0.559114,0.266319,0.311237,0.009676
2,-1.580936,-0.944257,-0.699556,0.925871,-1.667262,-0.655132,-0.1318,-0.171891,0.0,0.0,...,-1.564540,-0.883135,0.029629,-0.025941,1.124979,0.080403,0.560801,0.266319,0.311237,0.009676
3,-1.597680,-0.895144,-0.796059,0.925871,-1.659605,-0.655132,-0.1318,-0.171891,0.0,0.0,...,-1.556426,-0.883135,0.029629,-0.025941,1.119324,0.079541,0.559114,0.266319,0.311237,0.009676
4,-1.580936,-1.091596,0.554978,0.925871,-1.736177,-0.655132,-0.1318,-0.171891,0.0,0.0,...,-1.637565,-0.883135,0.029629,-0.025941,1.175877,0.088160,0.575986,0.266319,0.311237,0.009676


In [ ]:
# Task 2: Model building
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Split features and target
X = df.drop(columns=["Rented Bike Count"])
y = df["Rented Bike Count"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Elastic Net Regression": ElasticNet()
}

# Define parameter grids for tuning
param_grids = {
    "Ridge Regression": {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    "Lasso Regression": {"alpha": [0.001, 0.01, 0.1, 1.0, 10.0]},
    "Elastic Net Regression": {
        "alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    }
}

# Store evaluation results
results = {}

# Fit and evaluate each model
for name, model in models.items():
    if name in param_grids:
        grid = GridSearchCV(model, param_grids[name], cv=5, scoring='r2')
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
    else:
        best_model = model.fit(X_train, y_train)

    # Predictions and metrics
    y_pred = best_model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        "MAE": mae,
        "MSE": mse,
        "R2": r2,
        "Best Params": getattr(best_model, "get_params", lambda: {})()
    }

# Convert results to DataFrame for viewing
results_df = pd.DataFrame(results).T
print(results_df)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.215e+08, tolerance: 2.348e+05
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.256e+08, tolerance: 2.339e+05
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

                               MAE            MSE        R2  \
Linear Regression       318.922664  190586.326342   0.54257   
Ridge Regression        318.804896  190305.128765  0.543245   
Lasso Regression        318.241547  189354.826929  0.545526   
Elastic Net Regression  318.261117  189400.045169  0.545418   

                                                              Best Params  
Linear Regression       {'copy_X': True, 'fit_intercept': True, 'n_job...  
Ridge Regression        {'alpha': 0.01, 'copy_X': True, 'fit_intercept...  
Lasso Regression        {'alpha': 0.1, 'copy_X': True, 'fit_intercept'...  
Elastic Net Regression  {'alpha': 0.001, 'copy_X': True, 'fit_intercep...  


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.551e+08, tolerance: 2.914e+05
  model = cd_fast.enet_coordinate_descent(


In [5]:
# Task 3: Model building with polynomial features
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import joblib

# Select numerical columns to apply polynomial features
selected_numerical_cols = [
    'Temperature(°C)', 'Humidity(%)', 'Wind speed (m/s)', 'Visibility (10m)'
]

# Create polynomial features up to degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train[selected_numerical_cols])

# Rebuild X_train with polynomial features and remaining features
X_train_poly = pd.DataFrame(X_poly, columns=poly.get_feature_names_out(selected_numerical_cols))
X_train_full = pd.concat([X_train_poly, X_train.drop(columns=selected_numerical_cols).reset_index(drop=True)], axis=1)

# Apply same transformation to X_test
X_test_poly = poly.transform(X_test[selected_numerical_cols])
X_test_poly_df = pd.DataFrame(X_test_poly, columns=poly.get_feature_names_out(selected_numerical_cols))
X_test_full = pd.concat([X_test_poly_df, X_test.drop(columns=selected_numerical_cols).reset_index(drop=True)], axis=1)

# Train a linear regression model with polynomial features
poly_model = LinearRegression()
poly_model.fit(X_train_full, y_train)

# Evaluate the model
y_pred_poly = poly_model.predict(X_test_full)
mae_poly = mean_absolute_error(y_test, y_pred_poly)
mse_poly = mean_squared_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

# Save the polynomial model only if it outperforms previous models
best_linear_r2 = max(result["R2"] for result in results.values())
if r2_poly > best_linear_r2:
    joblib.dump(poly_model, "best_polynomial_model.pkl")
    best_model_saved = "Polynomial Linear Regression"
else:
    best_model_saved = max(results, key=lambda x: results[x]["R2"])

# Output results
poly_results = {
    "MAE": mae_poly,
    "MSE": mse_poly,
    "R2": r2_poly,
    "Model Saved As": best_model_saved
}

print(poly_results)


{'MAE': 318.7880479239571, 'MSE': 190280.53335446375, 'R2': 0.5433043283889701, 'Model Saved As': 'Lasso Regression'}


In [6]:
#Task 4: Model evaluation and validation
from sklearn.model_selection import cross_val_score, GridSearchCV
import numpy as np
import pandas as pd

# Set up 5-fold cross-validation
cv = 5

# Dictionary to store cross-validation results
cv_results = {}

# Evaluate original regression models (with and without regularization)
for name, model in models.items():
    if name in param_grids:
        grid = GridSearchCV(model, param_grids[name], cv=cv, scoring='r2')
        grid.fit(X, y)
        best_model = grid.best_estimator_
    else:
        best_model = model
    scores = cross_val_score(best_model, X, y, cv=cv, scoring='r2')
    cv_results[name] = {
        "CV Mean R2": np.mean(scores),
        "CV Std R2": np.std(scores)
    }

# Evaluate the polynomial model
X_poly_all = poly.transform(X[selected_numerical_cols])
X_poly_df = pd.DataFrame(X_poly_all, columns=poly.get_feature_names_out(selected_numerical_cols))
X_poly_full = pd.concat([X_poly_df, X.drop(columns=selected_numerical_cols).reset_index(drop=True)], axis=1)

poly_cv_scores = cross_val_score(poly_model, X_poly_full, y, cv=cv, scoring='r2')
cv_results["Polynomial Regression"] = {
    "CV Mean R2": np.mean(poly_cv_scores),
    "CV Std R2": np.std(poly_cv_scores)
}

# Display results
cv_results_df = pd.DataFrame(cv_results).T
print(cv_results_df)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.675e+08, tolerance: 3.093e+05
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.166e+08, tolerance: 3.198e+05
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

                        CV Mean R2  CV Std R2
Linear Regression        -1.696874   3.233567
Ridge Regression         -1.400402   2.876842
Lasso Regression         -1.088492   2.354718
Elastic Net Regression   -0.107293   0.541137
Polynomial Regression    -1.778935   3.385793


Analyzed bike rental data using several regression models to understand what affects rental demand. Among all the models, Lasso Regression gave the best results because it was both accurate and good at ignoring unimportant features.Also created new features by combining existing ones and tested a more complex model using polynomial features. However, this more complex model did not perform better than the simpler Lasso model. From the data, we found that temperature, humidity, sunlight, and whether it was a working day were important factors that influenced how many bikes were rented. For example, more bikes were rented on warmer, sunnier working days, while rentals dropped on days with high humidity or rain. These findings can help the business plan better, such as offering promotions during good weather or preparing for lower demand on rainy days. To improve the model further, we suggest trying tree based models, adding time related features like hour or weekday, and using extra data like local events or traffic. Also recommend creating a dashboard to monitor predictions and retraining the model regularly with new data.